In [ ]:
#pip install pandas
#pip install openpyxl
#pip install duckdb

import pandas as pd
import sqlite3
import duckdb


In [ ]:
'''-----Left join-----'''

# Read Excel files
product_df = pd.read_excel("product_sample_500.csv.xlsx", dtype=str)
links_df = pd.read_excel("links.csv.xlsx", dtype=str)

# Replace blanks with NULL
product_df = product_df.replace('', pd.NA)

# Create SQL memory DB
conn = sqlite3.connect(":memory:")

# Load tables
product_df.to_sql("product_sample_500", conn, index=False, if_exists="replace")
links_df.to_sql("links", conn, index=False, if_exists="replace")

# Overlay logic
query = """
SELECT 
    p.*,
    COALESCE(p.GMID_CD, l.GMID_CD) AS FILLED_GMID
FROM product_sample_500 p
LEFT JOIN links l
ON p.product_id = l.product_id
"""

# Execute
final_product = pd.read_sql_query(query, conn)

# Replace original GMID
final_product["GMID_CD"] = final_product["FILLED_GMID"]
final_product.drop(columns=["FILLED_GMID"], inplace=True)

# Save
final_product.to_excel("product_sample_500_cleaned.xlsx", index=False)

print("Cleaned file saved")

Cleaned file saved


In [ ]:
'''------Right join------'''

# Read Excel files
product_df = pd.read_excel("product_sample_500.csv.xlsx", dtype=str)
links_df = pd.read_excel("links.csv.xlsx", dtype=str)

# Replace blanks with NULL
product_df = product_df.replace('', pd.NA)
links_df = links_df.replace('', pd.NA)

# Connect to DuckDB
conn = duckdb.connect()

# Register dataframes as tables
conn.register("product_sample_500", product_df)
conn.register("links", links_df)

# RIGHT JOIN query
query = """
SELECT 
    l.*,
    COALESCE(l.GMID_CD, p.GMID_CD) AS FILLED_GMID
FROM product_sample_500 p
RIGHT JOIN links l
ON p.product_id = l.product_id
"""

# Execute query
final_links = conn.execute(query).df()

# Replace GMID column
final_links["GMID_CD"] = final_links["FILLED_GMID"]
final_links.drop(columns=["FILLED_GMID"], inplace=True)

# Save result
final_links.to_excel("links_cleaned.xlsx", index=False)

print("Cleaned links file saved")

Cleaned links file saved


In [ ]:
'''-----Full join-----'''

# Read Excel files
product_df = pd.read_excel("product_sample_500.csv.xlsx", dtype=str)
links_df = pd.read_excel("links.csv.xlsx", dtype=str)

# Replace blanks with NULL
product_df = product_df.replace('', pd.NA)
links_df = links_df.replace('', pd.NA)

# Connect DuckDB
conn = duckdb.connect()

# Register tables
conn.register("product", product_df)
conn.register("links", links_df)

# FULL OUTER JOIN query
query = """
SELECT *
FROM product p
FULL OUTER JOIN links l
ON p.product_id = l.product_id
"""

# Execute query
full_join = conn.execute(query).df()

# Save result
full_join.to_excel("Merged_file.xlsx", index=False)

print("Merged file saved")

Merged file saved
